# Free Cursor QLoRA Training (Colab T4)\n\nThis notebook runs the full pipeline end-to-end:\n1. Verify GPU\n2. Install dependencies\n3. Mount Google Drive\n4. Extract dataset archive\n5. Train QLoRA\n6. Evaluate\n7. Merge LoRA and export ONNX bundle\n\nExpected dataset archive in Drive (fallback supported):\n- `/content/drive/MyDrive/free_cursor_dataset.tar.gz`\n- `/content/drive/MyDrive/dataset.tar.gz`

In [ ]:
import os

REPO_URL = "https://github.com/dewd5252/free-cursor-mvp.git"
REPO_DIR = "/content/free-cursor-mvp"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/free-cursor-mvp
print("Repo ready at /content/free-cursor-mvp")

In [ ]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets sentencepiece optimum[onnxruntime] onnx onnxruntime

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import tarfile

archive_candidates = [
    '/content/drive/MyDrive/free_cursor_dataset.tar.gz',
    '/content/drive/MyDrive/dataset.tar.gz',
]

archive_path = next((p for p in archive_candidates if os.path.exists(p)), None)
if archive_path is None:
    raise FileNotFoundError(
        'Dataset archive not found. Upload free_cursor_dataset.tar.gz to MyDrive.'
    )

print('Using archive:', archive_path)

with tarfile.open(archive_path, 'r:gz') as tar:
    tar.extractall('/content')

for required in [
    '/content/dataset/splits/train.jsonl',
    '/content/dataset/splits/val.jsonl',
    '/content/dataset/splits/test.jsonl',
]:
    if not os.path.exists(required):
        raise FileNotFoundError(f'Missing expected file: {required}')

print('Dataset extracted successfully.')

In [ ]:
import os
# قائمة بالملفات الموجودة في الجذر لـ Google Drive للمساعدة في تحديد مكان الملف
drive_path = '/content/drive/MyDrive'
if os.path.exists(drive_path):
    print("Files in MyDrive:")
    for f in os.listdir(drive_path):
        if f.endswith('.gz') or 'dataset' in f.lower():
            print(f"- {f}")
else:
    print("Google Drive is not mounted. Please run the mount cell above.")

In [ ]:
import json
import pandas as pd
from datasets import Dataset, DatasetDict

paths = {
    'train': '/content/dataset/splits/train.jsonl',
    'validation': '/content/dataset/splits/val.jsonl',
    'test': '/content/dataset/splits/test.jsonl'
}

def load_manual(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return Dataset.from_pandas(pd.DataFrame(data))

try:
    ds = DatasetDict({
        'train': load_manual(paths['train']),
        'validation': load_manual(paths['validation']),
        'test': load_manual(paths['test'])
    })
    print("✅ Success! Manual load completed.")
    print(ds)
    # Check first record to confirm integrity
    print("\nFirst sample target:", ds['train'][0]['target'])
except Exception as e:
    print(f"❌ Manual load failed: {e}")

In [ ]:
!python scripts/ml/colab_train_qlora.py \
  --model-id Qwen/Qwen2.5-0.5B-Instruct \
  --train /content/dataset/splits/train.jsonl \
  --val /content/dataset/splits/val.jsonl \
  --test /content/dataset/splits/test.jsonl \
  --max-seq-length 1024 \
  --epochs 1 \
  --per-device-batch 4 \
  --grad-accum 4 \
  --lr 2e-4 \
  --lora-output /content/drive/MyDrive/free_cursor_lora_weights \
  --report-output /content/drive/MyDrive/free_cursor_eval_report.json

In [ ]:
import json

report_path = '/content/drive/MyDrive/free_cursor_eval_report.json'
with open(report_path, 'r', encoding='utf-8') as f:
    report = json.load(f)

print(json.dumps(report, ensure_ascii=False, indent=2))

In [ ]:
!python scripts/ml/merge_and_export_onnx.py \
  --base-model Qwen/Qwen2.5-0.5B-Instruct \
  --lora-dir /content/drive/MyDrive/free_cursor_lora_weights \
  --merged-dir /content/free_cursor_merged \
  --onnx-dir /content/drive/MyDrive/free_cursor_onnx_bundle

In [ ]:
import os\n\nbundle_dir = '/content/drive/MyDrive/free_cursor_onnx_bundle'\nrequired = [\n    'model.onnx',\n    'tokenizer.json',\n    'tokenizer_config.json',\n    'special_tokens_map.json',\n    'generation_config.json',\n]\n\nmissing = [name for name in required if not os.path.exists(os.path.join(bundle_dir, name))]\nif missing:\n    print('Missing files:', missing)\nelse:\n    print('ONNX bundle looks complete.')\n    for name in required:\n        path = os.path.join(bundle_dir, name)\n        print(f'- {name}: {os.path.getsize(path)} bytes')